In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_extraction.text import TfidfVectorizer

# Создаем синтетический датасет недвижимости
data = pd.DataFrame({
    'Rooms': [1, 2, 3, 1, 4, 2, 3, 1],
    'Area': [35.0, 55.0, 90.0, 30.0, 120.0, 60.0, 85.0, 40.0],
    'Has_Elevator': [1, 0, 1, 0, 1, 1, 0, 1],
    'Floor': [2, 5, 9, 4, 12, 3, 10, 1],
    'Description': [
        "Уютная однокомнатная квартира с евроремонтом. Тихий двор.",
        "Просторная двухкомнатная квартира. Требуется косметический ремонт!",
        "Роскошные трехкомнатные апартаменты! Отличный вид, евроремонт, панорамные окна.",
        "Срочная продажа! Маленькая квартира без ремонта. ТОРГ!",
        "Элитная 4-комнатная квартира в центре. Дизайнерский ремонт, паркинг, джакузи.",
        "Светлая квартира рядом с метро. Хороший ремонт, тихие соседи.",
        "Квартира на верхнем этаже. Красивый вид, без лифта, нужен ремонт.",
        "Компактная студия. Отличный вариант под сдачу в аренду!"
    ],
    'Price': [5_000_000, 7_500_000, 15_000_000, 4_100_000, 28_000_000, 8_900_000, 11_000_000, 5_800_000]
})

# Разделяем на train/test ДО трансформаций!
X = data.drop(columns=['Price'])
y = data['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [3]:
# 1
from sklearn.preprocessing import PolynomialFeatures

X_train['Area_per_Room'] = X_train['Area'] / X_train['Rooms']
X_test['Area_per_Room'] = X_test['Area'] / X_test['Rooms']

X_train['Floor_No_Elevator'] = X_train['Floor'] * (1 - X_train['Has_Elevator'])
X_test['Floor_No_Elevator'] = X_test['Floor'] * (1 - X_test['Has_Elevator'])

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_poly_train = poly.fit_transform(X_train[['Rooms', 'Area']])
X_poly_test = poly.transform(X_test[['Rooms', 'Area']])
# а как правильно сделать что бы сразу встроить новые столбцы в исходню матрицу?


In [7]:
# 2
X_train['desc_len'] = X_train['Description'].str.len()
X_test['desc_len'] = X_test['Description'].str.len()

X_train['has_exclamation'] = X_train['Description'].str.contains('!').astype(int)
X_test['has_exclamation'] = X_test['Description'].str.contains('!').astype(int)

# Первый способ
# pattern = r'\b[А-ЯЁA-Z]{2,}\b'

# X_train['has_capitals'] = X_train['Description'].str.contains(pattern, regex=True).astype(int)
# X_test['has_capitals'] = X_test['Description'].str.contains(pattern, regex=True).astype(int)

# Второй
def cheak_capitals(text):
    des = text.replace('!', '').replace('.', '').split()

    for word in des:
        if len(word)>1 and word.isupper():
            return 1
    return 0

X_train['has_capitals'] = X_train['Description'].apply(cheak_capitals)
X_test['has_capitals'] = X_test['Description'].apply(cheak_capitals)


In [ ]:
# 3
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5)
X_tfidf_train = vectorizer.fit_transform(X_train['Description'])
X_tfidf_test = vectorizer.transform(X_test['Description'])

tfidf_columns = vectorizer.get_feature_names_out()

df_tfidf_train = pd.DataFrame(X_tfidf_train.toarray(), columns=tfidf_columns, index=X_train.index)
df_tfidf_test = pd.DataFrame(X_tfidf_test.toarray(), columns=tfidf_columns, index=X_test.index)

X_train = pd.concat([X_train, df_tfidf_train], axis=1)
X_test = pd.concat([X_test, df_tfidf_test], axis=1)

X_train = X_train.drop(columns=['Description'])
X_test = X_test.drop(columns=['Description'])


Контрольные вопросы
1. iteractions_only=True чтобы столбцы не умножались сами на себя а только на другие столбцы; потому что появится очень много столбцов, они сразу заполнят всю оперативную память
2. тогда у X_test будут совершенно другие ключевые слова и их значения